# Augmentation Zone

This notebook is responsible of augmenting the data present on the dataset by performing different transformations. The goal is to obtain a larger dataset so that the model fine-tuning becomes more robust. 

Notice that this zone is placed after the labelling and splitting zone. That is because augmented data must only be performed on training data to avoid data leakage (when the original data and the transformation belong to train and test sets respectively).

The following code will do the following steps:
- Reading the dataset created in the splitting zone 
- Apply transformations to images while preserving the same label (or associated text) and store them in a new bucket
- Regenerating the dataset manifest for training set     

## Preparing new buckets

In [60]:
import boto3
from botocore.exceptions import ClientError
import os
from dotenv import load_dotenv

load_dotenv()
access_key_id = os.getenv("ACCESS_KEY_ID")
secret_access_key = os.getenv("SECRET_ACCESS_KEY")
minio_url = "http://" + os.getenv("S3_API_ENDPOINT")


minio_client = boto3.client(
    "s3",
    aws_access_key_id=access_key_id,
    aws_secret_access_key=secret_access_key,
    endpoint_url=minio_url
)

#manifest_name = "dataset_labelled.json"

In [61]:
new_bucket = "augmentation-zone"
try:
    minio_client.create_bucket(Bucket=new_bucket)
except ClientError as e:
    error_code = e.response['Error']['Code']
    if error_code in ['BucketAlreadyExists', 'BucketAlreadyOwnedByYou']:
        print(f"Bucket '{new_bucket}' already exists")

Bucket 'augmentation-zone' already exists


## Load training dataset

We will download the training dataset from the splitting zone to process only the training images and maintain the text-image relationships.


In [62]:
import json
import pandas as pd

# Descargar el dataset de train desde splitting-zone
source_bucket = "splitting-zone"
train_manifest_name = "dataset_train.json"

train_dataset = []

try:
    minio_client.download_file(source_bucket, train_manifest_name, train_manifest_name)
    print(f"Dataset de train descargado: {train_manifest_name}")
    
    # Leer el dataset
    with open(train_manifest_name, 'r') as f:
        train_dataset = json.load(f)

    print(train_dataset)

    print(f"Dataset cargado: {len(train_dataset)} entradas")
    if len(train_dataset) > 0:
        print(f"  Ejemplo de entrada: {train_dataset['image']['0']}")
except Exception as e:
    print(f"Error descargando el dataset: {e}")
    train_dataset = []

Dataset de train descargado: dataset_train.json
{'image': {'43': 'images/ISIC_0029220.png', '62': 'images/ISIC_0030492.png', '3': 'images/ISIC_0025118.png', '71': 'images/ISIC_0031981.png', '45': 'images/ISIC_0029475.png', '48': 'images/ISIC_0029649.png', '6': 'images/ISIC_0025298.png', '99': 'images/ISIC_0034222.png', '82': 'images/ISIC_0032731.png', '76': 'images/ISIC_0032218.png', '60': 'images/ISIC_0030413.png', '80': 'images/ISIC_0032701.png', '90': 'images/ISIC_0033464.png', '68': 'images/ISIC_0031655.png', '51': 'images/ISIC_0029851.png', '27': 'images/ISIC_0027219.png', '18': 'images/ISIC_0026289.png', '56': 'images/ISIC_0030197.png', '63': 'images/ISIC_0030541.png', '74': 'images/ISIC_0032110.png', '1': 'images/ISIC_0024508.png', '61': 'images/ISIC_0030421.png', '42': 'images/ISIC_0029024.png', '41': 'images/ISIC_0028948.png', '4': 'images/ISIC_0025200.png', '15': 'images/ISIC_0026128.png', '17': 'images/ISIC_0026283.png', '40': 'images/ISIC_0028726.png', '38': 'images/ISIC_00

In [63]:
source_bucket = "splitting-zone"

response = minio_client.list_objects_v2(Bucket=source_bucket)
if 'Contents' in response:
    for obj in response['Contents']:
        copy_source = {'Bucket': source_bucket, 'Key': obj['Key']}
        minio_client.copy_object(CopySource=copy_source, Bucket=new_bucket, Key=obj['Key'])
        print(f"Copied {obj['Key']} from {source_bucket} to {new_bucket}")
else:
    print(f"No objects found in bucket '{source_bucket}'.")

Copied dataset_labelled.json from splitting-zone to augmentation-zone
Copied dataset_test.json from splitting-zone to augmentation-zone
Copied dataset_train.json from splitting-zone to augmentation-zone
Copied images/ISIC_0024388.png from splitting-zone to augmentation-zone
Copied images/ISIC_0024508.png from splitting-zone to augmentation-zone
Copied images/ISIC_0024853.png from splitting-zone to augmentation-zone
Copied images/ISIC_0025118.png from splitting-zone to augmentation-zone
Copied images/ISIC_0025200.png from splitting-zone to augmentation-zone
Copied images/ISIC_0025202.png from splitting-zone to augmentation-zone
Copied images/ISIC_0025298.png from splitting-zone to augmentation-zone
Copied images/ISIC_0025343.png from splitting-zone to augmentation-zone
Copied images/ISIC_0025430.png from splitting-zone to augmentation-zone
Copied images/ISIC_0025806.png from splitting-zone to augmentation-zone
Copied images/ISIC_0025874.png from splitting-zone to augmentation-zone
Copie

## Data augmentation

Here, we transform the images (data augmentation) while keeping the embedding of the original image, so that both share the same semantic representation.

The available transformations are the following:
- Rotation
- Bright alteration
- Contrast alteration
- Mirroring (Flip horizontally)

Except for the latest one, all the degree of the transformation (e.g. the rotation or increasing of brightness) is selected at random. The selection of which transformations are applied to each image in the training set is also performed at random:

In [64]:
import random

random.seed(0) # Fix the seed

def apply_transformations(image):
    """Aplica transformaciones aleatorias a la imagen"""
    images_transformations = []
    transformations = []
    
    # Rotación aleatoria
    if random.random() > 0.5:
        angle = random.randint(-15, 15)
        images_transformations.append(image.rotate(angle, expand=False, fillcolor='white'))
        transformations.append(f"rotated_{angle}")
    
    # Ajuste de brillo
    if random.random() > 0.5:
        enhancer = ImageEnhance.Brightness(image)
        factor = random.uniform(0.8, 1.2)
        images_transformations.append(enhancer.enhance(factor))
        transformations.append(f"brightness_{factor:.2f}")
    
    # Ajuste de contraste
    if random.random() > 0.5:
        enhancer = ImageEnhance.Contrast(image)
        factor = random.uniform(0.8, 1.2)
        images_transformations.append(enhancer.enhance(factor))
        transformations.append(f"contrast_{factor:.2f}")
    
    # Flip horizontal
    if random.random() > 0.5:
        images_transformations.append(ImageOps.mirror(image))
        transformations.append("flipped")

    print(images_transformations)
    
    return images_transformations, transformations

In [65]:
import io
from PIL import Image, ImageEnhance, ImageOps
import random
import chromadb
import time
import json

# Buckets
source_bucket_images = "splitting-zone"  # Donde están las imágenes originales
destination_bucket = "augmentation-zone"

# Inicializar el dataset aumentado
augmented_dataset = []
ids_disponibles = list(train_dataset['image'].keys())

print(f"Procesando {len(ids_disponibles)} entradas del dataset de train...")
print("=" * 80)

if len(ids_disponibles) > 0:
    transformed_count = 0
    
    for entry_idx, id_key in enumerate(ids_disponibles):
        try:
            image_key = train_dataset['image'].get(id_key, '')
            text_key = train_dataset['text'].get(id_key, '')
            
            if not image_key or not text_key:
                print(f"Entrada {id_key} sin imagen o texto, saltando...")
                continue
            
            print(f"\n[{entry_idx+1}/{len(ids_disponibles)}] Procesando ID {id_key}: {image_key} -> {text_key}")
            
            # Intentar descargar la imagen desde diferentes buckets
            image_obj = minio_client.get_object(Bucket=source_bucket_images, Key=image_key)
            image_data = image_obj['Body'].read()
            print(f"Imagen descargada desde {source_bucket_images}")
            
            if image_data is None:
                print(f"No se pudo descargar la imagen {image_key} desde ningún bucket")
                continue
            
            # Cargar la imagen
            original_image = Image.open(io.BytesIO(image_data)).convert('RGB')
            
            # Añadir la imagen original al dataset aumentado
            augmented_dataset.append({
                "image": image_key,
                "text": text_key
            })
            
            # Aplicar transformaciones
            transformed_images, transformations = apply_transformations(original_image.copy())

            for i, transformed_image in enumerate(transformed_images):
                # Crear nombre para la imagen transformada
                base_name = image_key.replace('images/', '').replace('.png', '').replace('.jpg', '').replace('.jpeg', '')
                name = transformations[i]
                transform_suffix = f"_{name}" if transformations[i] else 'augmented'
                new_key = f"images/{base_name}_{transform_suffix}.png"
                
                # Guardar imagen transformada
                buffer = io.BytesIO()
                transformed_image.save(buffer, format='PNG')
                buffer_data = buffer.getvalue()
                
                # Subir imagen transformada a MinIO
                minio_client.put_object(
                    Bucket=destination_bucket,
                    Key=new_key,
                    Body=buffer_data,
                    ContentType='image/png'
                )
                print(f"Imagen transformada guardada: {new_key}")
                
                # Añadir la imagen transformada al dataset aumentado con el mismo texto
                augmented_dataset.append({
                    "image": new_key,
                    "text": text_key  # Mismo texto que la original
                })
                
                transformed_count += 1
                print(f"Transformaciones aplicadas: {', '.join(transformations) if transformations else 'ninguna'}")
            
        except Exception as e:
            print(f"Error procesando entrada {entry_idx}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    print(f"\n{'='*80}")
    print(f"Proceso completado:")
    print(f"Entradas procesadas: {transformed_count}")
    print(f"Total de entradas en dataset aumentado: {len(augmented_dataset)}")
    print(f"Entradas originales: {len(train_dataset)}")
    print(f"Entradas nuevas (transformadas): {len(augmented_dataset) - len(train_dataset)}")
else:
    print("No hay datos en el dataset de train para procesar.")



Procesando 80 entradas del dataset de train...

[1/80] Procesando ID 43: images/ISIC_0029220.png -> texts/melanoma_0_0_74.txt
Imagen descargada desde splitting-zone
[<PIL.Image.Image image mode=RGB size=600x400 at 0x18FEF7BBF90>, <PIL.Image.Image image mode=RGB size=600x400 at 0x18FEF86A790>, <PIL.Image.Image image mode=RGB size=600x400 at 0x18FEF86BD10>, <PIL.Image.Image image mode=RGB size=600x400 at 0x18FEF838290>]
Imagen transformada guardada: images/ISIC_0029220__rotated_9.png
Transformaciones aplicadas: rotated_9, brightness_0.82, contrast_0.99, flipped
Imagen transformada guardada: images/ISIC_0029220__brightness_0.82.png
Transformaciones aplicadas: rotated_9, brightness_0.82, contrast_0.99, flipped
Imagen transformada guardada: images/ISIC_0029220__contrast_0.99.png
Transformaciones aplicadas: rotated_9, brightness_0.82, contrast_0.99, flipped
Imagen transformada guardada: images/ISIC_0029220__flipped.png
Transformaciones aplicadas: rotated_9, brightness_0.82, contrast_0.99, fl

## Prove the validation of the new images transforme matching the same Embedding.

Now we're going to print all the embeddings with all the images that have associated, so we can validate that the augmentation part have been done successfully.

In [66]:
print(augmented_dataset)

[{'image': 'images/ISIC_0029220.png', 'text': 'texts/melanoma_0_0_74.txt'}, {'image': 'images/ISIC_0029220__rotated_9.png', 'text': 'texts/melanoma_0_0_74.txt'}, {'image': 'images/ISIC_0029220__brightness_0.82.png', 'text': 'texts/melanoma_0_0_74.txt'}, {'image': 'images/ISIC_0029220__contrast_0.99.png', 'text': 'texts/melanoma_0_0_74.txt'}, {'image': 'images/ISIC_0029220__flipped.png', 'text': 'texts/melanoma_0_0_74.txt'}, {'image': 'images/ISIC_0030492.png', 'text': 'texts/basal_cell_carcinoma_0_0_47.txt'}, {'image': 'images/ISIC_0030492__rotated_15.png', 'text': 'texts/basal_cell_carcinoma_0_0_47.txt'}, {'image': 'images/ISIC_0030492__contrast_1.16.png', 'text': 'texts/basal_cell_carcinoma_0_0_47.txt'}, {'image': 'images/ISIC_0030492__flipped.png', 'text': 'texts/basal_cell_carcinoma_0_0_47.txt'}, {'image': 'images/ISIC_0025118.png', 'text': 'texts/melanoma_0_0_74.txt'}, {'image': 'images/ISIC_0025118__brightness_1.05.png', 'text': 'texts/melanoma_0_0_74.txt'}, {'image': 'images/ISI

In [67]:
## Guardar el dataset aumentado

# Guardar el dataset aumentado en un archivo JSON
local_filename_augmented = "dataset_train_augmented.json"

with open(local_filename_augmented, "w") as f:
    json.dump(augmented_dataset, f, indent=2)

print(f"✓ Dataset aumentado guardado localmente: {local_filename_augmented}")
print(f"  Total de entradas: {len(augmented_dataset)}")

# Subir el dataset aumentado a MinIO
minio_client.upload_file(local_filename_augmented, new_bucket, local_filename_augmented)
print(f"✓ Dataset aumentado subido a {new_bucket}/{local_filename_augmented}")

# También guardar una copia del dataset original de train en augmentation-zone
local_filename_train = "dataset_train.json"
with open(local_filename_train, "w") as f:
    json.dump(train_dataset, f, indent=2)
minio_client.upload_file(local_filename_train, new_bucket, local_filename_train)
print(f"✓ Dataset original de train también guardado en {new_bucket}/{local_filename_train}")

print(f"\nResumen final:")
print(f"  - Dataset original (train): {len(train_dataset)} entradas")
print(f"  - Dataset aumentado: {len(augmented_dataset)} entradas")
print(f"  - Incremento: {len(augmented_dataset) - len(train_dataset)} nuevas entradas ({((len(augmented_dataset) - len(train_dataset)) / len(train_dataset) * 100):.1f}% más)")

✓ Dataset aumentado guardado localmente: dataset_train_augmented.json
  Total de entradas: 249
✓ Dataset aumentado subido a augmentation-zone/dataset_train_augmented.json
✓ Dataset original de train también guardado en augmentation-zone/dataset_train.json

Resumen final:
  - Dataset original (train): 4 entradas
  - Dataset aumentado: 249 entradas
  - Incremento: 245 nuevas entradas (6125.0% más)


In [68]:
from IPython.display import display
df = pd.read_json(local_filename_augmented)
df.describe()
display(df)

,image,text
0,images/ISIC_0029220.png,texts/melanoma_0_0_74.txt
1,images/ISIC_0029220__rotated_9.png,texts/melanoma_0_0_74.txt
2,images/ISIC_0029220__brightness_0.82.png,texts/melanoma_0_0_74.txt
3,images/ISIC_0029220__contrast_0.99.png,texts/melanoma_0_0_74.txt
4,images/ISIC_0029220__flipped.png,texts/melanoma_0_0_74.txt
...,...,...
244,images/ISIC_0029623__rotated_-6.png,texts/melanoma_0_0_47.txt
245,images/ISIC_0029623__brightness_0.95.png,texts/melanoma_0_0_47.txt
246,images/ISIC_0029263.png,texts/basal_cell_carcinoma_0_0_13.txt
247,images/ISIC_0029263__rotated_-8.png,texts/basal_cell_carcinoma_0_0_13.txt
